# Xay dung Lap chi muc ngu nghia tiem an

In [ ]:
import os
import nltk
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from whoosh.index import create_in
from whoosh.fields import *
from whoosh.analysis import StandardAnalyzer
from whoosh import qparser
from whoosh import scoring
import whoosh.index as index

import pytrec_eval
import math


nltk.download('punkt_tab')
nltk.download('stopwords')
stoplist = stopwords.words("english")
stoplist.append('oh')
puncts = ['.', ',', ':', '`', '"', "'", '!', '?', "``", "''"]
ps = PorterStemmer()
import shutil

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
def preprocess(tok, stemmer=ps, punctlist=puncts, stopwords=stoplist):
  tok = tok.lower()
  if tok.isdigit():
    return None
  if tok.isnumeric():
    return None
  if tok in punctlist:
    return None
  if tok in stopwords:
    return None
  return stemmer.stem(tok)

def indexing(src, idx="ind"):
  if os.path.exists(idx):
      shutil.rmtree(idx)
  os.mkdir(idx)
  if src[-1] != '/':
    src += '/'
  schema = Schema(docid=STORED(), content=TEXT(stored=True, analyzer=StandardAnalyzer()))
  ix = create_in(idx, schema)
  writer = ix.writer()

  files = os.listdir(src)
  for f in files:
    r = open(src + f, encoding="cp1252")
    terms = []
    for s in r:
      for sent in sent_tokenize(s.strip()):
        for tok in word_tokenize(sent):
          tok = preprocess(tok)
          if tok != None:
            terms.append(tok)
    r.close()
    cont = " ".join(terms)
    writer.add_document(docid="{}".format(f.split(".")[0]), content=cont)
  writer.commit()

In [3]:
indexing("Cranfield/Cranfield", "ind")

In [4]:
def readGroundTruth(src):
  if src[-1] != '/':
    src += '/'

  GT = {}
  for f in os.listdir(src):
    r = open(src + f)
    rel = {}
    for s in r:
      s = s.strip()
      sp = s.split("\t")
      if len(sp) < 2:
        continue
      did = sp[0].split(" ")[1]
      rel[did] = int(sp[1])
    GT[f.split(".")[0]] = rel
    r.close()
  return GT

In [5]:
GroundTruth = readGroundTruth("Cranfield/RES")
print(GroundTruth)

{'1': {'184': 2, '29': 2, '31': 2, '12': 3, '51': 3, '102': 3, '13': 4, '14': 4, '15': 4, '57': 2, '378': 2, '859': 2, '185': 3, '30': 3, '37': 3, '52': 4, '142': 4, '195': 4, '875': 2, '56': 3, '66': 3, '95': 3, '462': 4, '497': 3, '858': 3, '876': 3, '879': 3, '880': 3, '486': -1}, '10': {'259': 2, '405': 2, '302': 3, '436': 3, '437': 3, '438': 3, '998': 3, '1011': 3, '493': -1}, '100': {'821': 2, '822': 2, '824': 2, '820': 3, '823': 3, '825': 3, '1122': 2, '1051': 3, '1121': 3, '760': -1}, '101': {'817': 2, '818': 2, '819': 2, '820': 3, '825': 3, '824': 2, '760': -1}, '102': {'728': 1, '913': 2, '910': 3, '911': 4, '729': -1}, '103': {'826': 3, '828': 3, '761': -1}, '104': {'833': 3, '834': 3, '835': 3, '836': 3, '837': 3, '762': -1}, '105': {'848': 3, '844': 4, '845': 4, '846': 4, '847': 4, '764': -1}, '106': {'847': 2, '846': 3, '849': 3, '844': 4, '845': 4, '764': -1}, '107': {'725': 2, '728': 2, '729': 3, '911': 3, '720': 4, '75': 4, '909': 4, '640': -1}, '108': {'724': 2, '726'

In [6]:
def readQuery(src):
  qry = {}
  r = open(src)
  for s in r:
    s = s.strip()
    ps = s.split("\t")
    terms = []
    for sent in sent_tokenize(ps[1]):
      for tok in word_tokenize(sent):
        tok = preprocess(tok)
        if tok != None:
          terms.append(tok)
    qry[ps[0]] = " ".join(terms)
  return qry

In [7]:
Queries = readQuery("Cranfield/query.txt")
print(Queries)

{'1': 'similar law must obey construct aeroelast model heat high speed aircraft', '2': 'structur aeroelast problem associ flight high speed aircraft', '3': 'problem heat conduct composit slab solv far', '4': 'criterion develop show empir valid flow solut chemic react ga mixtur base simplifi assumpt instantan local chemic equilibrium', '5': 'chemic kinet system applic hyperson aerodynam problem', '6': 'theoret experiment guid turbul couett flow behaviour', '7': 'possibl relat avail pressur distribut ogiv forebodi zero angl attack lower surfac pressur equival ogiv forebodi angl attack', '8': 'method -dash exact approxim -dash present avail predict bodi pressur angl attack', '9': 'paper intern /slip flow/ heat transfer studi', '10': 'real-ga transport properti air avail wide rang enthalpi densiti', '11': 'possibl find analyt similar solut strong blast wave problem newtonian approxim', '12': 'aerodynam perform channel flow ground effect machin calcul', '13': 'basic mechan transon aileron b

In [10]:
from gensim import corpora, models, similarities

def readDocuments(src):
    docs = {}
    if src[-1] != '/':
        src += '/'
    for f in os.listdir(src):
        r = open(src + f, encoding="cp1252")
        terms = []
        for s in r:
            for sent in sent_tokenize(s.strip()):
                for tok in word_tokenize(sent):
                    tok = preprocess(tok)
                    if tok is not None:
                        terms.append(tok)
        r.close()
        docs[f.split(".")[0]] = terms
    return docs

documents = readDocuments("Cranfield/Cranfield")
print(len(documents))


1400


In [11]:
dictionary = corpora.Dictionary(documents.values())
corpus = [dictionary.doc2bow(doc) for doc in documents.values()]
doc_ids = list(documents.keys())


In [12]:
tfidf = models.TfidfModel(corpus)
corpus_tfidf = tfidf[corpus]


In [14]:
NUM_TOPICS = 200

lsi = models.LsiModel(
    corpus_tfidf,
    id2word=dictionary,
    num_topics=NUM_TOPICS
)

corpus_lsi = lsi[corpus_tfidf]


In [15]:
index_lsi = similarities.MatrixSimilarity(
    corpus_lsi,
    num_features=NUM_TOPICS
)


In [16]:
def processQueries(qry, dictionary, tfidf, lsi, index_lsi, doc_ids, topk=1000):
    RunResults = {}

    for qid, qtext in qry.items():
        q_tokens = qtext.split()
        q_bow = dictionary.doc2bow(q_tokens)
        q_tfidf = tfidf[q_bow]
        q_lsi = lsi[q_tfidf]

        sims = index_lsi[q_lsi]  # cosine similarity
        ranked = sorted(enumerate(sims), key=lambda x: -x[1])

        RunResults[qid] = {}
        for rank, (doc_idx, score) in enumerate(ranked[:topk]):
            if score > 0:
                RunResults[qid][doc_ids[doc_idx]] = float(score)

    return RunResults


In [18]:
Queries = readQuery("Cranfield/query.txt")
RunResults = processQueries(
    Queries,
    dictionary,
    tfidf,
    lsi,
    index_lsi,
    doc_ids
)


In [30]:
evaluator = pytrec_eval.RelevanceEvaluator(
    GroundTruth,
    {
        "map",
        "infAP",
        "11pt_avg",
        "P_5",
        "P_10",
        "P_20",
        "P_30",
        "P_50",
        "recall_5",
        "recall_10",
        "recall_20",
        "recall_30",
        "recall_50",
    }
)

results = evaluator.evaluate(RunResults)

for qid, metrics in results.items():
    print(f"Query {qid}")
    for m, v in metrics.items():
        print(f"  {m:10s}: {v:.4f}")
    print("-" * 30)


Query 1
  map       : 0.2697
  P_5       : 0.6000
  P_10      : 0.6000
  P_20      : 0.4500
  P_30      : 0.3000
  P_50      : 0.2000
  recall_5  : 0.1071
  recall_10 : 0.2143
  recall_20 : 0.3214
  recall_30 : 0.3214
  recall_50 : 0.3571
  infAP     : 0.3264
  11pt_avg  : 0.2988
------------------------------
Query 2
  map       : 0.2098
  P_5       : 0.6000
  P_10      : 0.3000
  P_20      : 0.2500
  P_30      : 0.2333
  P_50      : 0.1400
  recall_5  : 0.1250
  recall_10 : 0.1250
  recall_20 : 0.2083
  recall_30 : 0.2917
  recall_50 : 0.2917
  infAP     : 0.2115
  11pt_avg  : 0.2446
------------------------------
Query 3
  map       : 0.4487
  P_5       : 0.4000
  P_10      : 0.5000
  P_20      : 0.3000
  P_30      : 0.2000
  P_50      : 0.1200
  recall_5  : 0.3333
  recall_10 : 0.8333
  recall_20 : 1.0000
  recall_30 : 1.0000
  recall_50 : 1.0000
  infAP     : 0.5918
  11pt_avg  : 0.5227
------------------------------
Query 4
  map       : 0.7500
  P_5       : 0.4000
  P_10      : 

In [32]:
MAP = InfMAP = MAP11 = P5 = P10 = P20 = P30 = P50 = Recall5 = Recall10 = Recall20 = Recall30 = Recall50 = 0
cnt = 0

for qid in results:
    r = results[qid]
    if math.isnan(r["infAP"]):
        continue

    MAP += r["map"]
    InfMAP += r["infAP"]
    MAP11 += r["11pt_avg"]
    P5 += r["P_5"]
    P10 += r["P_10"]
    P20 += r["P_20"]
    P30 += r["P_30"]
    P50 += r["P_50"]
    Recall5 += r["recall_5"]
    Recall10 += r["recall_10"]
    Recall20 += r["recall_20"]
    Recall30 += r["recall_30"]
    Recall50 += r["recall_50"]

    cnt += 1


In [34]:
print(f"MAP      : {MAP / cnt:.4f}")
print(f"InfMAP   : {InfMAP / cnt:.4f}")
print(f"MAP11PT  : {MAP11 / cnt:.4f}")
print(f"P@5      : {P5 / cnt:.4f}")
print(f"P@10     : {P10 / cnt:.4f}")
print(f"P@20     : {P20 / cnt:.4f}")
print(f"P@30     : {P30 / cnt:.4f}")
print(f"P@50     : {P50 / cnt:.4f}")
print(f"Recall@5 : {Recall5 / cnt:.4f}")
print(f"Recall@10: {Recall10 / cnt:.4f}")
print(f"Recall@20: {Recall20 / cnt:.4f}")
print(f"Recall@30: {Recall30 / cnt:.4f}")
print(f"P@50     : {P50 / cnt:.4f}")

MAP      : 0.2998
InfMAP   : 0.3497
MAP11PT  : 0.3223
P@5      : 0.2942
P@10     : 0.2462
P@20     : 0.1669
P@30     : 0.1279
P@50     : 0.0883
Recall@5 : 0.2572
Recall@10: 0.4114
Recall@20: 0.5353
Recall@30: 0.5985
P@50     : 0.0883


In [ ]:
print(pytrec_eval.supported_measures)
eval = pytrec_eval.RelevanceEvaluator(GroundTruth, ["infAP", "11pt_avg"])

{'relstring', 'map', 'G', 'gm_bpref', 'gm_map', 'map_cut', 'Rndcg', 'P', 'Rprec', 'set_map', 'iprec_at_recall', 'num_rel', 'success', 'set_P', 'runid', 'num_nonrel_judged_ret', 'num_ret', 'num_rel_ret', 'utility', '11pt_avg', 'recall', 'relative_P', 'recip_rank', 'set_F', 'ndcg_cut', 'ndcg_rel', 'num_q', 'set_recall', 'bpref', 'set_relative_P', 'binG', 'Rprec_mult', 'infAP', 'ndcg'}
0.349729154610263 0.32225425268665747


## Lemma

In [39]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

In [40]:
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN


In [41]:
from nltk import pos_tag

def preprocess(tok, punctlist=puncts, stopwords=stoplist):
    tok = tok.lower()

    if tok.isdigit() or tok.isnumeric():
        return None
    if tok in punctlist:
        return None
    if tok in stopwords:
        return None

    pos = pos_tag([tok])[0][1]
    wn_pos = get_wordnet_pos(pos)

    return lemmatizer.lemmatize(tok, wn_pos)

In [56]:
import shutil
import os

def indexing(src, idx="ind"):
    if os.path.exists(idx):
        shutil.rmtree(idx)

    os.mkdir(idx)

    schema = Schema(
        docid=STORED(),
        content=TEXT(stored=True, analyzer=StandardAnalyzer())
    )

    ix = create_in(idx, schema)
    writer = ix.writer()

    total_terms = 0          # tổng số token
    vocab = set()            # từ vựng (unique terms)

    if src[-1] != '/':
        src += '/'

    for f in os.listdir(src):
        with open(src + f, encoding="cp1252") as r:
            terms = []
            for s in r:
                for sent in sent_tokenize(s.strip()):
                    for tok in word_tokenize(sent):
                        tok = preprocess(tok)
                        if tok is not None:
                            terms.append(tok)
                            total_terms += 1
                            vocab.add(tok)

        writer.add_document(
            docid=f.split(".")[0],
            content=" ".join(terms)
        )

    writer.commit()

    print("Total terms (tokens):", total_terms)
    print("Vocabulary size (unique terms):", len(vocab))


In [57]:
indexing("Cranfield/Cranfield", "ind")

Total terms (tokens): 131697
Vocabulary size (unique terms): 6394


In [46]:
GroundTruth = readGroundTruth("Cranfield/RES")
print(GroundTruth)

{'1': {'184': 2, '29': 2, '31': 2, '12': 3, '51': 3, '102': 3, '13': 4, '14': 4, '15': 4, '57': 2, '378': 2, '859': 2, '185': 3, '30': 3, '37': 3, '52': 4, '142': 4, '195': 4, '875': 2, '56': 3, '66': 3, '95': 3, '462': 4, '497': 3, '858': 3, '876': 3, '879': 3, '880': 3, '486': -1}, '10': {'259': 2, '405': 2, '302': 3, '436': 3, '437': 3, '438': 3, '998': 3, '1011': 3, '493': -1}, '100': {'821': 2, '822': 2, '824': 2, '820': 3, '823': 3, '825': 3, '1122': 2, '1051': 3, '1121': 3, '760': -1}, '101': {'817': 2, '818': 2, '819': 2, '820': 3, '825': 3, '824': 2, '760': -1}, '102': {'728': 1, '913': 2, '910': 3, '911': 4, '729': -1}, '103': {'826': 3, '828': 3, '761': -1}, '104': {'833': 3, '834': 3, '835': 3, '836': 3, '837': 3, '762': -1}, '105': {'848': 3, '844': 4, '845': 4, '846': 4, '847': 4, '764': -1}, '106': {'847': 2, '846': 3, '849': 3, '844': 4, '845': 4, '764': -1}, '107': {'725': 2, '728': 2, '729': 3, '911': 3, '720': 4, '75': 4, '909': 4, '640': -1}, '108': {'724': 2, '726'

In [47]:
Queries = readQuery("Cranfield/query.txt")
print(Queries)

{'1': 'similarity law must obeyed construct aeroelastic model heat high speed aircraft', '2': 'structural aeroelastic problem associate flight high speed aircraft', '3': 'problem heat conduction composite slab solve far', '4': 'criterion developed show empirically validity flow solution chemically react gas mixture base simplify assumption instantaneous local chemical equilibrium', '5': 'chemical kinetic system applicable hypersonic aerodynamic problem', '6': 'theoretical experimental guide turbulent couette flow behaviour', '7': 'possible relate available pressure distribution ogive forebody zero angle attack low surface pressure equivalent ogive forebody angle attack', '8': 'method -dash exact approximate -dash presently available predict body pressure angle attack', '9': 'paper internal /slip flow/ heat transfer study', '10': 'real-gas transport property air available wide range enthalpy density', '11': 'possible find analytical similar solution strong blast wave problem newtonian a

In [48]:
documents = readDocuments("Cranfield/Cranfield")
print(len(documents))

1400


In [49]:
dictionary = corpora.Dictionary(documents.values())
corpus = [dictionary.doc2bow(doc) for doc in documents.values()]
doc_ids = list(documents.keys())


In [50]:
tfidf = models.TfidfModel(corpus)
corpus_tfidf = tfidf[corpus]


In [51]:
NUM_TOPICS = 200

lsi = models.LsiModel(
    corpus_tfidf,
    id2word=dictionary,
    num_topics=NUM_TOPICS
)

corpus_lsi = lsi[corpus_tfidf]


In [52]:
index_lsi = similarities.MatrixSimilarity(
    corpus_lsi,
    num_features=NUM_TOPICS
)


In [58]:
def readQuery(src):
  qry = {}
  r = open(src)
  for s in r:
    s = s.strip()
    ps = s.split("\t")
    terms = []
    for sent in sent_tokenize(ps[1]):
      for tok in word_tokenize(sent):
        tok = preprocess(tok)
        if tok != None:
          terms.append(tok)
    qry[ps[0]] = " ".join(terms)
  return qry

In [59]:
Queries = readQuery("Cranfield/query.txt")
RunResults = processQueries(
    Queries,
    dictionary,
    tfidf,
    lsi,
    index_lsi,
    doc_ids
)


In [60]:
evaluator = pytrec_eval.RelevanceEvaluator(
    GroundTruth,
    {
        "map",
        "infAP",
        "11pt_avg",
        "P_5",
        "P_10",
        "P_20",
        "P_30",
        "P_50",
        "recall_5",
        "recall_10",
        "recall_20",
        "recall_30",
        "recall_50",
    }
)

results = evaluator.evaluate(RunResults)

for qid, metrics in results.items():
    print(f"Query {qid}")
    for m, v in metrics.items():
        print(f"  {m:10s}: {v:.4f}")
    print("-" * 30)


Query 1
  map       : 0.2230
  P_5       : 0.6000
  P_10      : 0.5000
  P_20      : 0.3500
  P_30      : 0.2667
  P_50      : 0.2200
  recall_5  : 0.1071
  recall_10 : 0.1786
  recall_20 : 0.2500
  recall_30 : 0.2857
  recall_50 : 0.3929
  infAP     : 0.2699
  11pt_avg  : 0.2341
------------------------------
Query 2
  map       : 0.1669
  P_5       : 0.4000
  P_10      : 0.3000
  P_20      : 0.2000
  P_30      : 0.1667
  P_50      : 0.1200
  recall_5  : 0.0833
  recall_10 : 0.1250
  recall_20 : 0.1667
  recall_30 : 0.2083
  recall_50 : 0.2500
  infAP     : 0.1690
  11pt_avg  : 0.1913
------------------------------
Query 3
  map       : 0.5148
  P_5       : 0.6000
  P_10      : 0.6000
  P_20      : 0.3000
  P_30      : 0.2000
  P_50      : 0.1200
  recall_5  : 0.5000
  recall_10 : 1.0000
  recall_20 : 1.0000
  recall_30 : 1.0000
  recall_50 : 1.0000
  infAP     : 0.6736
  11pt_avg  : 0.6000
------------------------------
Query 4
  map       : 1.0000
  P_5       : 0.4000
  P_10      : 

In [61]:
print(f"MAP      : {MAP / cnt:.4f}")
print(f"InfMAP   : {InfMAP / cnt:.4f}")
print(f"MAP11PT  : {MAP11 / cnt:.4f}")
print(f"P@5      : {P5 / cnt:.4f}")
print(f"P@10     : {P10 / cnt:.4f}")
print(f"P@20     : {P20 / cnt:.4f}")
print(f"P@30     : {P30 / cnt:.4f}")
print(f"P@50     : {P50 / cnt:.4f}")
print(f"Recall@5 : {Recall5 / cnt:.4f}")
print(f"Recall@10: {Recall10 / cnt:.4f}")
print(f"Recall@20: {Recall20 / cnt:.4f}")
print(f"Recall@30: {Recall30 / cnt:.4f}")
print(f"P@50     : {P50 / cnt:.4f}")

MAP      : 0.2998
InfMAP   : 0.3497
MAP11PT  : 0.3223
P@5      : 0.2942
P@10     : 0.2462
P@20     : 0.1669
P@30     : 0.1279
P@50     : 0.0883
Recall@5 : 0.2572
Recall@10: 0.4114
Recall@20: 0.5353
Recall@30: 0.5985
P@50     : 0.0883
